In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS erp_lakehouse.gold")

In [0]:
orders = spark.table("erp_lakehouse.silver.orders")
lineitem = spark.table("erp_lakehouse.silver.lineitem")
customer = spark.table("erp_lakehouse.silver.customer")
part = spark.table("erp_lakehouse.silver.part")

fact_sales = (lineitem
    .join(orders, lineitem.l_orderkey == orders.o_orderkey, "inner")
    .join(customer, orders.o_custkey == customer.c_custkey, "inner")
    .join(part, lineitem.l_partkey == part.p_partkey, "inner")
    .select(
        orders.o_orderkey,
        orders.o_orderdate,
        F.year("o_orderdate").alias("order_year"),
        F.month("o_orderdate").alias("order_month"),
        customer.c_custkey,
        customer.c_name.alias("customer_name"),
        customer.c_nationkey,
        part.p_partkey,
        part.p_name.alias("part_name"),
        part.p_brand,
        lineitem.l_quantity,
        lineitem.l_extendedprice,
        lineitem.l_discount,
        (lineitem.l_extendedprice * (1 - lineitem.l_discount)).alias("net_revenue")
    ))

(fact_sales.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("order_year", "order_month")
    .saveAsTable("erp_lakehouse.gold.fact_sales"))

print(f"✅ gold.fact_sales: {fact_sales.count()} linhas")

In [0]:
sales_summary = (spark.table("erp_lakehouse.gold.fact_sales")
    .groupBy("order_year", "order_month", "c_nationkey")
    .agg(
        F.sum("net_revenue").alias("total_revenue"),
        F.sum("l_quantity").alias("total_quantity"),
        F.countDistinct("o_orderkey").alias("total_orders"),
        F.countDistinct("c_custkey").alias("total_customers")
    )
    .orderBy("order_year", "order_month"))

(sales_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("erp_lakehouse.gold.sales_summary_monthly"))

print(f"✅ gold.sales_summary_monthly: {sales_summary.count()} linhas")

In [0]:
spark.sql("OPTIMIZE erp_lakehouse.gold.fact_sales ZORDER BY (c_nationkey, p_brand)")

In [0]:
spark.table("erp_lakehouse.gold.fact_sales").printSchema()